In [1]:
!pip uninstall -y prophet fbprophet pystan cmdstanpy
!pip install cmdstanpy==1.1.0
import cmdstanpy
cmdstanpy.install_cmdstan()
!pip install prophet


Found existing installation: prophet 1.1.7
Uninstalling prophet-1.1.7:
  Successfully uninstalled prophet-1.1.7
Found existing installation: cmdstanpy 1.3.0
Uninstalling cmdstanpy-1.3.0:
  Successfully uninstalled cmdstanpy-1.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.2/83.2 kB 2.8 MB/s eta 0:00:00
Installing CmdStan version: 2.37.0
Install directory: /root/.cmdstan
Download successful, file: /tmp/tmpywafux8h
Extracting distribution


DEBUG:cmdstanpy:cmd: make build -j1
cwd: None


Unpacked download as cmdstan-2.37.0
Building version cmdstan-2.37.0, may take several minutes, depending on your system.


DEBUG:cmdstanpy:cmd: make examples/bernoulli/bernoulli
cwd: None


Test model compilation
Installed cmdstan-2.37.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 119.4 MB/s eta 0:00:00


In [4]:
import matplotlib
# Force matplotlib to not use any Xwindows backend
matplotlib.use('Agg')

import pandas as pd
from prophet import Prophet
from prophet.serialize import model_to_json, model_from_json
import matplotlib.pyplot as plt
import numpy as np
import os
import hashlib
import json
import scipy.stats as stats
from sklearn.metrics import mean_squared_error, r2_score
from scipy.signal import find_peaks

# ---------------------------------------------------------
# 1. CONFIGURATION (CONFIGURACIÓN)
# ---------------------------------------------------------
FILE_PATH = 'weather-air-quality-clean.csv.bz2'
CACHE_DIR = "model_cache"
OUTPUT_DIR = "analysis_output"

LOCKDOWN_START = '2020-03-20'
LOCKDOWN_END_FOR_MASKING = '2020-05-10'

TEST_PERIODS = [
    ('2020-03-20', '2020-04-12'),
    ('2020-03-20', '2020-05-10')
]

TARGET_GROUPS = {
    'pm10_median': ['pm10_centenario', 'pm10_cordoba', 'pm10_la_boca'],
    'no2_median':  ['no2_centenario', 'no2_cordoba', 'no2_la_boca'],
    'co_median':   ['co_centenario', 'co_cordoba', 'co_la_boca', 'co_palermo']
}

# Nombres para gráficos
DISPLAY_NAMES = {
    'pm10_median': 'PM10 (Mediana)',
    'no2_median':  'NO2 (Mediana)',
    'co_median':   'CO (Mediana)'
}

REGRESSOR_NAMES = [
    'reg_temperature', 'reg_relativehumidity', 'reg_pressure',
    'reg_windspeed', 'reg_precipitation', 'reg_wind_sin', 'reg_wind_cos'
]

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------------------------------------------------
# 2. HELPER FUNCTIONS (Plotting in Spanish)
# ---------------------------------------------------------
def create_diagnostics_panel(target_key, results_df, train_indices, output_dir):
    """
    Generates a 6-panel diagnostic plot (Spanish). PNG ONLY (300 DPI).
    Includes Residuals vs Predicted and Standardized Residuals.
    """
    target_display = DISPLAY_NAMES.get(target_key, target_key)
    df_train = results_df.loc[train_indices].dropna(subset=['y', 'yhat'])

    if df_train.empty:
        return

    actuals = df_train['y']
    preds = df_train['yhat']
    residuals = actuals - preds

    # Calculate Standardized Residuals (Z-score)
    residuals_std = (residuals - residuals.mean()) / residuals.std()

    # Create 3 rows x 2 columns layout
    fig, axes = plt.subplots(3, 2, figsize=(16, 18))
    fig.suptitle(f'Diagnóstico Avanzado: {target_display}', fontsize=16)

    # --- ROW 1: BASIC FIT ---
    # 1. Actual vs Predicted
    ax = axes[0, 0]
    ax.scatter(actuals, preds, alpha=0.3, s=10, color='blue')
    min_val = min(actuals.min(), preds.min())
    max_val = max(actuals.max(), preds.max())
    ax.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', label='Ajuste Perfecto')
    ax.set_title('Observado vs. Predicho')
    ax.set_xlabel('Observación Real')
    ax.set_ylabel('Predicción del Modelo')
    ax.grid(True, alpha=0.3)

    # 2. Residuals over Time
    ax = axes[0, 1]
    ax.scatter(df_train['ds'], residuals, alpha=0.3, s=10, color='purple')
    ax.axhline(0, color='black', linestyle='--')
    ax.set_title('Residuos en el Tiempo (Estacionariedad)')
    ax.set_ylabel('Residuo (Real - Predicho)')
    ax.set_xlabel('Fecha')
    ax.grid(True, alpha=0.3)

    # --- ROW 2: HOMOSCEDASTICITY ---
    # 3. Residuals vs Predicted
    ax = axes[1, 0]
    ax.scatter(preds, residuals, alpha=0.3, s=10, color='teal')
    ax.axhline(0, color='black', linestyle='--')
    ax.set_title('Residuos vs. Predichos (Homocedasticidad)')
    ax.set_xlabel('Valor Predicho')
    ax.set_ylabel('Residuo')
    ax.grid(True, alpha=0.3)

    # 4. Standardized Residuals vs Predicted
    ax = axes[1, 1]
    ax.scatter(preds, residuals_std, alpha=0.3, s=10, color='darkorange')
    ax.axhline(0, color='black', linestyle='-')
    ax.axhline(2, color='red', linestyle='--', alpha=0.5, label='±2 SD')
    ax.axhline(-2, color='red', linestyle='--', alpha=0.5)
    ax.axhline(3, color='red', linestyle=':', alpha=0.5, label='±3 SD')
    ax.axhline(-3, color='red', linestyle=':', alpha=0.5)
    ax.set_title('Residuos Estandarizados vs. Predichos')
    ax.set_xlabel('Valor Predicho')
    ax.set_ylabel('Residuo Estandarizado (Z-Score)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # --- ROW 3: DISTRIBUTION ---
    # 5. Residual Histogram
    ax = axes[2, 0]
    ax.hist(residuals, bins=50, color='green', alpha=0.7, density=True)
    ax.set_title('Distribución de Residuos')
    ax.set_xlabel('Magnitud del Error')
    ax.set_ylabel('Frecuencia')
    ax.grid(True, alpha=0.3)

    # 6. Q-Q Plot
    ax = axes[2, 1]
    stats.probplot(residuals, dist="norm", plot=ax)
    ax.set_title('Gráfico Q-Q (Normalidad)')
    ax.set_xlabel('Cuantiles Teóricos')
    ax.set_ylabel('Valores Ordenados')
    ax.grid(True, alpha=0.3)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

    # Save to PNG with 300 DPI
    base_path = os.path.join(output_dir, f"diagnostics_panel_{target_key}")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close(fig)
    print(f"  -> Panel de diagnóstico avanzado guardado: {base_path}.png")

# ---------------------------------------------------------
# 3. LOAD DATA
# ---------------------------------------------------------
print("Cargando datos...")
try:
    df = pd.read_csv(FILE_PATH)
    df['ds'] = pd.to_datetime(df['date']).dt.tz_localize(None)
except FileNotFoundError:
    print(f"Error: No se encontró el archivo {FILE_PATH}")
    exit(1)

# ---------------------------------------------------------
# 4. WEATHER REGRESSOR PREPARATION
# ---------------------------------------------------------
print("Procesando variables meteorológicas...")

# A. Linear Variables
linear_recipes = [
    ('reg_temperature',      ['temperature_observatorio', 'temperature_aeroparque'],       'temperature_openmeteo'),
    ('reg_relativehumidity', ['relativehumidity_observatorio', 'relativehumidity_aeroparque'], 'relativehumidity_openmeteo'),
    ('reg_pressure',         ['pressure_observatorio', 'pressure_aeroparque'],             'pressure_openmeteo'),
    ('reg_windspeed',        ['windspeed_observatorio', 'windspeed_aeroparque'],           'windspeed_openmeteo'),
    ('reg_precipitation',    ['precipitation_observatorio', 'precipitation_aeroparque'],   'precipitation_openmeteo')
]

for new_col, local_cols, fallback_col in linear_recipes:
    valid_locals = [c for c in local_cols if c in df.columns]
    if valid_locals:
        df[new_col] = df[valid_locals].mean(axis=1)
    else:
        df[new_col] = np.nan

    if fallback_col in df.columns:
        df[new_col] = df[new_col].fillna(df[fallback_col])

    df[new_col] = df[new_col].interpolate(method='linear').ffill().bfill()

# B. Circular Variables
wind_cols = ['windangle_observatorio', 'windangle_aeroparque', 'windangle_openmeteo']
for col in wind_cols:
    if col in df.columns:
        rads = df[col] * (np.pi / 180)
        df[f'{col}_sin'] = np.sin(rads)
        df[f'{col}_cos'] = np.cos(rads)

def fuse_components(suffix):
    locals_comp = [f'{c}_{suffix}' for c in ['windangle_observatorio', 'windangle_aeroparque'] if f'{c}_{suffix}' in df.columns]
    fallback_comp = f'windangle_openmeteo_{suffix}'
    if locals_comp:
        series = df[locals_comp].mean(axis=1)
    else:
        series = pd.Series(np.nan, index=df.index)
    if fallback_comp in df.columns:
        series = series.fillna(df[fallback_comp])
    return series.interpolate(method='linear').ffill().bfill()

df['reg_wind_sin'] = fuse_components('sin')
df['reg_wind_cos'] = fuse_components('cos')

# ---------------------------------------------------------
# 5. POLLUTION TARGET AGGREGATION
# ---------------------------------------------------------
print("Calculando medianas de contaminantes (Solo estaciones locales)...")
for target_name, source_cols in TARGET_GROUPS.items():
    valid_cols = [c for c in source_cols if c in df.columns]
    if valid_cols:
        df[target_name] = df[valid_cols].median(axis=1)

# ---------------------------------------------------------
# 6. ANALYSIS LOOP
# ---------------------------------------------------------
for target_col in TARGET_GROUPS.keys():
    if target_col not in df.columns: continue

    # Get pretty name for display
    target_display = DISPLAY_NAMES.get(target_col, target_col)

    if df[target_col].notna().sum() < 100:
        print(f"Saltando {target_display}: Datos insuficientes.")
        continue

    print(f"\n{'='*40}")
    print(f"Analizando Objetivo: {target_display}")
    print(f"{'='*40}")

    cols = ['ds', target_col] + REGRESSOR_NAMES
    data = df[cols].copy()
    data.rename(columns={target_col: 'y'}, inplace=True)

    # --- MASKING ---
    model_df = data.copy()
    mask_intervention = (model_df['ds'] >= LOCKDOWN_START) & (model_df['ds'] <= LOCKDOWN_END_FOR_MASKING)
    model_df.loc[mask_intervention, 'y'] = None

    # --- CACHING ---
    params = {
        'daily': True, 'weekly': True, 'yearly': True, 'prior': 0.05,
        'regressors': sorted(REGRESSOR_NAMES),
        'target': target_col,
        'lockdown_start': LOCKDOWN_START,
        'lockdown_end': LOCKDOWN_END_FOR_MASKING
    }

    param_str = json.dumps(params, sort_keys=True)
    param_hash = hashlib.md5(param_str.encode("utf-8")).hexdigest()
    data_sig = f"{model_df['ds'].min()}-{model_df['ds'].max()}-{len(model_df)}-{model_df['y'].sum()}"
    data_hash = hashlib.md5(data_sig.encode("utf-8")).hexdigest()

    cache_key = f"{target_col}_{param_hash[:6]}_{data_hash[:6]}"
    cache_file = os.path.join(CACHE_DIR, f"prophet_{cache_key}.json")

    if os.path.exists(cache_file):
        print(f"Cache Hit! Cargando modelo desde {cache_file}...")
        with open(cache_file, 'r') as fin:
            m = model_from_json(json.load(fin))
    else:
        print(f"Cache Miss. Entrenando modelo...")
        m = Prophet(
            daily_seasonality=params['daily'],
            weekly_seasonality=params['weekly'],
            yearly_seasonality=params['yearly'],
            changepoint_prior_scale=params['prior']
        )
        for reg in REGRESSOR_NAMES:
            m.add_regressor(reg)

        m.fit(model_df)

        with open(cache_file, 'w') as fout:
            json.dump(model_to_json(m), fout)

    # -----------------------------------------------------
    # PREDICTION & DIAGNOSTICS
    # -----------------------------------------------------
    print("Generando predicciones contrafácticas...")
    forecast = m.predict(data)
    results = pd.merge(data[['ds', 'y']], forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']], on='ds')

    train_indices = model_df.dropna(subset=['y']).index
    y_true = model_df.loc[train_indices, 'y']
    y_pred = forecast.loc[train_indices, 'yhat']

    if len(y_true) > 0:
        r2 = r2_score(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        residuals = y_true - y_pred

        # Text Report
        report_path = os.path.join(OUTPUT_DIR, f"diagnostics_{target_col}.txt")
        with open(report_path, "w") as f:
            f.write(f"REPORTE DE DIAGNÓSTICO PARA {target_display}\n")
            f.write(f"====================================\n")
            f.write(f"R-Cuadrado (Varianza Explicada): {r2:.4f}\n")
            f.write(f"RMSE (Error Medio):              {rmse:.4f}\n")
            f.write(f"Sesgo Medio (Residual Mean):     {residuals.mean():.4f}\n")

        # B. Plots (Spanish, PNG only, 300 DPI)
        create_diagnostics_panel(target_col, results, train_indices, OUTPUT_DIR)

        # C. Components (PNG only, 300 DPI)
        fig_comp = m.plot_components(forecast)
        comp_base = os.path.join(OUTPUT_DIR, f"components_{target_col}")
        fig_comp.savefig(f"{comp_base}.png", dpi=300)
        plt.close(fig_comp)

        # D. [REMOVED] Standalone Residuals Histogram (Now in Panel)

    # -----------------------------------------------------
    # IMPACT PLOTTING (SPANISH)
    # -----------------------------------------------------
    plot_start = '2020-02-15'
    plot_end = '2020-06-15'
    mask_plot = (results['ds'] >= plot_start) & (results['ds'] <= plot_end)
    plot_data = results[mask_plot]

    if not plot_data.empty:
        fig, ax = plt.subplots(figsize=(14, 7))

        # Plot translated labels
        ax.fill_between(plot_data['ds'], plot_data['yhat_lower'], plot_data['yhat_upper'],
                         color='gray', alpha=0.3, label='Rango Esperado (Sin ASPO)')
        ax.plot(plot_data['ds'], plot_data['yhat'], color='#333333', linestyle='--', label='Media Esperada')

        ax.plot(plot_data['ds'], plot_data['y'], color='red', marker='o', markersize=3,
                linestyle='-', linewidth=1, alpha=0.7, label='Observado Real (Mediana)')

        ax.axvspan(pd.to_datetime(LOCKDOWN_START), pd.to_datetime(LOCKDOWN_END_FOR_MASKING),
                    color='green', alpha=0.15, label='ASPO (Aislamiento Estricto)')

        ax.set_title(f'Análisis de Impacto: {target_display}', fontsize=14)
        ax.set_ylabel('Concentración')
        ax.set_xlabel('Fecha')
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)

        # Impact plots: PNG (300 DPI) + SVG
        base_name = os.path.join(OUTPUT_DIR, f"impact_{target_col}")
        plt.savefig(f"{base_name}.png", dpi=300, bbox_inches='tight')
        plt.savefig(f"{base_name}.svg", bbox_inches='tight')
        print(f"  -> Gráficos de impacto guardados en {OUTPUT_DIR}")
        plt.close(fig)

    # -----------------------------------------------------
    # STATISTICAL OUTPUT
    # -----------------------------------------------------
    print("Calculando estadísticas de impacto...")
    for start_date, end_date in TEST_PERIODS:
        mask = (results['ds'] >= start_date) & (results['ds'] <= end_date)
        period_data = results[mask].dropna(subset=['y'])

        if len(period_data) == 0: continue

        mean_actual = period_data['y'].mean()
        mean_predicted = period_data['yhat'].mean()

        pct_change = 0
        if mean_predicted != 0:
            pct_change = ((mean_actual - mean_predicted) / mean_predicted) * 100

        print(f"  Periodo {start_date} al {end_date}:")
        print(f"    Esperado: {mean_predicted:.2f} | Observado: {mean_actual:.2f}")
        print(f"    Diferencia: {pct_change:.2f}%")

print("\nAnálisis Completado.")


Cargando datos...
Procesando variables meteorológicas...
Calculando medianas de contaminantes (Solo estaciones locales)...

Analizando Objetivo: PM10 (Mediana)
Cache Miss. Entrenando modelo...


DEBUG:cmdstanpy:input tempfile: /tmp/tmpt3xtuqbj/al6gatda.json
DEBUG:cmdstanpy:input tempfile: /tmp/tmpt3xtuqbj/kcs50b3z.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['/usr/local/lib/python3.12/dist-packages/prophet/stan_model/prophet_model.bin', 'random', 'seed=95417', 'data', 'file=/tmp/tmpt3xtuqbj/al6gatda.json', 'init=/tmp/tmpt3xtuqbj/kcs50b3z.json', 'output', 'file=/tmp/tmpt3xtuqbj/prophet_modelib8w4rcp/prophet_model-20251124134500.csv', 'method=optimize', 'algorithm=lbfgs', 'iter=10000']
13:45:00 - cmdstanpy - INFO - Chain [1] start processing
INFO:cmdstanpy:Chain [1] start processing
13:48:40 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing


Generando predicciones contrafácticas...
  -> Panel de diagnóstico avanzado guardado: analysis_output/diagnostics_panel_pm10_median.png
  -> Gráficos de impacto guardados en analysis_output
Calculando estadísticas de impacto...
  Periodo 2020-03-20 al 2020-04-12:
    Esperado: 19.33 | Observado: 13.15
    Diferencia: -32.00%
  Periodo 2020-03-20 al 2020-05-10:
    Esperado: 19.29 | Observado: 13.74
    Diferencia: -28.76%

Analizando Objetivo: NO2 (Mediana)
Cache Miss. Entrenando modelo...


DEBUG:cmdstanpy:input tempfile: /tmp/tmpt3xtuqbj/ovy4xri9.json
DEBUG:cmdstanpy:input tempfile: /tmp/tmpt3xtuqbj/yp36a612.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['/usr/local/lib/python3.12/dist-packages/prophet/stan_model/prophet_model.bin', 'random', 'seed=41572', 'data', 'file=/tmp/tmpt3xtuqbj/ovy4xri9.json', 'init=/tmp/tmpt3xtuqbj/yp36a612.json', 'output', 'file=/tmp/tmpt3xtuqbj/prophet_modelz5z4a66k/prophet_model-20251124134940.csv', 'method=optimize', 'algorithm=lbfgs', 'iter=10000']
13:49:40 - cmdstanpy - INFO - Chain [1] start processing
INFO:cmdstanpy:Chain [1] start processing
13:53:50 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing


Generando predicciones contrafácticas...
  -> Panel de diagnóstico avanzado guardado: analysis_output/diagnostics_panel_no2_median.png
  -> Gráficos de impacto guardados en analysis_output
Calculando estadísticas de impacto...
  Periodo 2020-03-20 al 2020-04-12:
    Esperado: 18.92 | Observado: 10.11
    Diferencia: -46.60%
  Periodo 2020-03-20 al 2020-05-10:
    Esperado: 20.08 | Observado: 13.16
    Diferencia: -34.46%

Analizando Objetivo: CO (Mediana)
Cache Miss. Entrenando modelo...


DEBUG:cmdstanpy:input tempfile: /tmp/tmpt3xtuqbj/aihm6mit.json
DEBUG:cmdstanpy:input tempfile: /tmp/tmpt3xtuqbj/xiwpi5l5.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['/usr/local/lib/python3.12/dist-packages/prophet/stan_model/prophet_model.bin', 'random', 'seed=40417', 'data', 'file=/tmp/tmpt3xtuqbj/aihm6mit.json', 'init=/tmp/tmpt3xtuqbj/xiwpi5l5.json', 'output', 'file=/tmp/tmpt3xtuqbj/prophet_modele8w1hd0c/prophet_model-20251124135454.csv', 'method=optimize', 'algorithm=lbfgs', 'iter=10000']
13:54:54 - cmdstanpy - INFO - Chain [1] start processing
INFO:cmdstanpy:Chain [1] start processing
13:56:30 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing


Generando predicciones contrafácticas...
  -> Panel de diagnóstico avanzado guardado: analysis_output/diagnostics_panel_co_median.png
  -> Gráficos de impacto guardados en analysis_output
Calculando estadísticas de impacto...
  Periodo 2020-03-20 al 2020-04-12:
    Esperado: 0.49 | Observado: 0.31
    Diferencia: -36.31%
  Periodo 2020-03-20 al 2020-05-10:
    Esperado: 0.53 | Observado: 0.35
    Diferencia: -34.43%

Análisis Completado.


In [8]:
def print_forecast_summary(results, target_col, lockdown_start, lockdown_end,
                           last_train_date='2020-05-10', display_names=None):

    if display_names is None:
        display_names = {target_col: target_col}

    target_display = display_names.get(target_col, target_col)

    lockdown_start = pd.to_datetime(lockdown_start)
    lockdown_end = pd.to_datetime(lockdown_end)
    last_train = pd.to_datetime(last_train_date)

    print("\n" + "="*60)
    print(f"RESUMEN COMPLETO: {target_display}")
    print("="*60)

    # ---------------------------------------------------------
    # Helper function for metrics
    # ---------------------------------------------------------
    def compute_metrics(df):
        mae = np.abs(df['y'] - df['yhat']).mean()
        rmse = np.sqrt(((df['y'] - df['yhat'])**2).mean())
        mape = (np.abs((df['y'] - df['yhat']) / df['yhat']).mean()) * 100
        return mae, rmse, mape

    # ---------------------------------------------------------
    # 1. PRE-ASPO
    # ---------------------------------------------------------
    pre = results[results['ds'] < lockdown_start].dropna(subset=['y'])
    if len(pre) > 0:
        mae, rmse, mape = compute_metrics(pre)

        print("\n1) PRE-ASPO (Entrenamiento)")
        print(f"   Observaciones: {len(pre)}")
        print(f"   MAE:  {mae:.3f}")
        print(f"   RMSE: {rmse:.3f}")
        print(f"   MAPE: {mape:.2f}%")

    # ---------------------------------------------------------
    # 2. ASPO (contrafactual)
    # ---------------------------------------------------------
    aspo = results[(results['ds'] >= lockdown_start) &
                   (results['ds'] <= lockdown_end)].dropna(subset=['y'])

    if len(aspo) > 0:
        mae, rmse, mape = compute_metrics(aspo)

        pct = ((aspo['y'].mean() - aspo['yhat'].mean()) /
               aspo['yhat'].mean()) * 100
        outside_ci = ((aspo['y'] < aspo['yhat_lower']) |
                      (aspo['y'] > aspo['yhat_upper'])).sum()

        print("\n2) ASPO (Contrafactual)")
        print(f"   Reducción estimada: {pct:.2f}%")
        print(f"   Puntos fuera del IC: {outside_ci}/{len(aspo)}")
        print(f"   MAE:  {mae:.3f}")
        print(f"   RMSE: {rmse:.3f}")
        print(f"   MAPE: {mape:.2f}%")

    # ---------------------------------------------------------
    # 3. POST-entrenamiento (forecast real)
    # ---------------------------------------------------------
    post = results[results['ds'] > last_train].dropna(subset=['y'])
    if len(post) > 0:
        mae, rmse, mape = compute_metrics(post)

        print("\n3) POST–Entrenamiento (Forecast Real)")
        print(f"   Observaciones: {len(post)}")
        print(f"   MAE:  {mae:.3f}")
        print(f"   RMSE: {rmse:.3f}")
        print(f"   MAPE: {mape:.2f}%")

    print("\n" + "="*60 + "\n")


In [10]:
plot_full_forecast_timeline(
    results=results,
    target_col='pm10_median',
    lockdown_start=LOCKDOWN_START,
    lockdown_end=LOCKDOWN_END_FOR_MASKING,
    last_train_date='2020-05-10',
    output_dir=OUTPUT_DIR,
    display_names=DISPLAY_NAMES
)

plot_full_forecast_with_zoom(
    results=results,
    target_col='pm10_median',
    lockdown_start=LOCKDOWN_START,
    lockdown_end=LOCKDOWN_END_FOR_MASKING,
    last_train_date='2020-05-10',
    output_dir=OUTPUT_DIR,
    display_names=DISPLAY_NAMES
)

print_forecast_summary(
    results=results,
    target_col='pm10_median',
    lockdown_start=LOCKDOWN_START,
    lockdown_end=LOCKDOWN_END_FOR_MASKING,
    last_train_date='2020-05-10',
    display_names=DISPLAY_NAMES
)


→ Timeline completo guardado en: analysis_output/full_timeline_pm10_median
→ Timeline con zoom guardado en: analysis_output/full_timeline_zoom_pm10_median

RESUMEN COMPLETO: PM10 (Mediana)

1) PRE-ASPO (Entrenamiento)
   Observaciones: 89447
   MAE:  0.225
   RMSE: 1.114
   MAPE: 43.37%

2) ASPO (Contrafactual)
   Reducción estimada: -34.43%
   Puntos fuera del IC: 0/1201
   MAE:  0.202
   RMSE: 0.228
   MAPE: 37.34%

3) POST–Entrenamiento (Forecast Real)
   Observaciones: 44805
   MAE:  0.163
   RMSE: 0.607
   MAPE: 37.09%




In [11]:
# revisa rangos y medias
print(results[['y','yhat']].describe())

# métricas con denom = y (MAPE clásica) y sMAPE por seguridad
def metrics_table(df):
    df = df.dropna(subset=['y','yhat'])
    mae = np.mean(np.abs(df['y'] - df['yhat']))
    rmse = np.sqrt(np.mean((df['y'] - df['yhat'])**2))
    # MAPE clásica: protejo división por cero
    mask = df['y'] != 0
    mape = np.mean(np.abs((df.loc[mask,'y'] - df.loc[mask,'yhat']) / df.loc[mask,'y'])) * 100
    # sMAPE
    denom = (np.abs(df['y']) + np.abs(df['yhat']))
    mask2 = denom != 0
    smape = np.mean(2 * np.abs(df.loc[mask2,'y'] - df.loc[mask2,'yhat']) / denom[mask2]) * 100
    return mae, rmse, mape, smape

for name, mask in [
    ('PRE-ASPO', results['ds'] < LOCKDOWN_START),
    ('ASPO', (results['ds'] >= LOCKDOWN_START) & (results['ds'] <= LOCKDOWN_END_FOR_MASKING)),
    ('POST', results['ds'] > pd.to_datetime('2020-05-10'))
]:
    df = results[mask]
    mae, rmse, mape, smape = metrics_table(df)
    print(f"{name}: n={len(df)}  MAE={mae:.3f}  RMSE={rmse:.3f}  MAPE(y denom)={mape:.2f}%  sMAPE={smape:.2f}%")


                   y           yhat
count  135453.000000  140592.000000
mean        0.530327       0.529378
std         0.987965       0.182686
min         0.000000      -0.124074
25%         0.330000       0.401890
50%         0.450000       0.518213
75%         0.610000       0.648272
max        44.730000       1.343151
PRE-ASPO: n=91752  MAE=0.225  RMSE=1.114  MAPE(y denom)=54.00%  sMAPE=38.11%
ASPO: n=1225  MAE=0.202  RMSE=0.228  MAPE(y denom)=70.08%  sMAPE=47.66%
POST: n=47615  MAE=0.163  RMSE=0.607  MAPE(y denom)=35.54%  sMAPE=30.67%


In [2]:
def create_complete_fourier_interpretation():
    """
    Crea un análisis de Fourier completo con interpretación desde cero
    """
    print("CREANDO ANÁLISIS FOURIER COMPLETO CON INTERPRETACIÓN")
    print("=" * 60)

    # Configuración
    FILE_PATH = 'weather-air-quality-clean.csv.bz2'
    OUTPUT_DIR = "fourier_analysis_complete"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Variables objetivo
    TARGET_GROUPS = {
        'pm10_median': ['pm10_centenario', 'pm10_cordoba', 'pm10_la_boca'],
        'no2_median':  ['no2_centenario', 'no2_cordoba', 'no2_la_boca'],
        'co_median':   ['co_centenario', 'co_cordoba', 'co_la_boca', 'co_palermo']
    }

    DISPLAY_NAMES = {
        'pm10_median': 'Material Particulado PM10',
        'no2_median':  'Dióxido de Nitrógeno (NO2)',
        'co_median':   'Monóxido de Carbono (CO)'
    }

    try:
        # Cargar datos
        print("Cargando datos...")
        df = pd.read_csv(FILE_PATH)
        df['ds'] = pd.to_datetime(df['date']).dt.tz_localize(None)

        # Calcular medianas
        print("Calculando medianas...")
        for target_name, source_cols in TARGET_GROUPS.items():
            valid_cols = [c for c in source_cols if c in df.columns]
            if valid_cols:
                df[target_name] = df[valid_cols].median(axis=1)

        # Reporte completo
        full_report = []
        full_report.append("REPORTE COMPLETO DE ANÁLISIS FOURIER - INTERPRETACIÓN")
        full_report.append("=" * 70)
        full_report.append("\n")

        # Analizar cada variable
        for target_col in TARGET_GROUPS.keys():
            if target_col not in df.columns:
                continue

            print(f"\nAnalizando {target_col}...")
            friendly_name = DISPLAY_NAMES.get(target_col, target_col)

            # Preparar datos
            data = df[['ds', target_col]].dropna()
            if len(data) < 100:
                print(f"  -> Saltando: solo {len(data)} puntos válidos")
                continue

            series = data[target_col].values
            dates = data['ds'].values

            # Análisis Fourier
            interpretation = analyze_series_fourier(friendly_name, series, dates, target_col, OUTPUT_DIR)
            if interpretation:
                full_report.extend(interpretation)
                full_report.append("\n" + "═" * 80 + "\n")

        # Guardar reporte principal
        report_path = os.path.join(OUTPUT_DIR, "REPORTE_INTERPRETACION_FOURIER.txt")
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(full_report))

        print(f"\n✅ REPORTE COMPLETO GUARDADO: {report_path}")

        # Mostrar resumen
        print("\n" + "="*70)
        print("RESUMEN EJECUTIVO DEL ANÁLISIS FOURIER")
        print("="*70)

        for line in full_report[:30]:  # Mostrar primeras 30 líneas
            print(line)

        return full_report

    except Exception as e:
        print(f"❌ Error en el análisis: {e}")
        import traceback
        traceback.print_exc()
        return []

def analyze_series_fourier(friendly_name, series, dates, target_key, output_dir):
    """
    Realiza análisis Fourier completo e interpretación para una serie
    """

    try:
        interpretation = []
        interpretation.append(f"🎯 ANÁLISIS DE {friendly_name.upper()}")
        interpretation.append("=" * 50)

        # Calcular sampling rate - manera más robusta
        if len(dates) > 1:
            time_diffs = np.diff(dates)
            median_diff_hours = np.median(time_diffs) / np.timedelta64(1, 'h')
            sampling_rate = 1.0 / median_diff_hours if median_diff_hours > 0 else 1.0
        else:
            sampling_rate = 1.0  # Valor por defecto

        # Calcular período total en días
        if len(dates) > 0:
            total_timedelta = dates[-1] - dates[0]
            total_days = total_timedelta / np.timedelta64(1, 'D')
        else:
            total_days = 0

        # Análisis Fourier
        n = len(series)
        if n < 10:
            interpretation.append("  ✗ Datos insuficientes para análisis Fourier")
            return interpretation

        detrended = series - np.mean(series)  # Remover media

        # FFT
        fft_result = np.fft.fft(detrended)
        frequencies = np.fft.fftfreq(n, d=1/sampling_rate)

        # Tomar parte positiva
        positive_freq_idx = frequencies > 0
        positive_freq = frequencies[positive_freq_idx]
        magnitude = np.abs(fft_result[positive_freq_idx])
        psd = (magnitude ** 2) / (n * sampling_rate)

        # Encontrar picos principales en frecuencias bajas
        low_freq_mask = positive_freq <= 1.0  # Frecuencias bajas (períodos > 1 hora)
        low_freq = positive_freq[low_freq_mask]
        low_psd = psd[low_freq_mask]

        if len(low_psd) > 0:
            # Encontrar picos con parámetros más flexibles
            min_height = np.percentile(low_psd, 80)  # Más bajo para detectar más picos
            peaks, properties = find_peaks(low_psd, height=min_height, distance=5)
        else:
            peaks = []

        # Interpretación
        interpretation.append(f"\n📊 DATOS GENERALES:")
        interpretation.append(f"  • Muestras analizadas: {len(series)}")
        interpretation.append(f"  • Frecuencia de muestreo: {sampling_rate:.2f} Hz")
        interpretation.append(f"  • Período total: {total_days:.1f} días")
        interpretation.append(f"  • Picos espectrales identificados: {len(peaks)}")

        # Patrones dominantes
        interpretation.append(f"\n🔍 PATRONES TEMPORALES IDENTIFICADOS:")

        if len(peaks) > 0:
            top_peaks = sorted(zip(peaks, low_psd[peaks]), key=lambda x: x[1], reverse=True)[:5]

            for i, (peak_idx, power) in enumerate(top_peaks, 1):
                freq = low_freq[peak_idx]
                period_hours = 1.0 / freq if freq > 0 else 0
                period_days = period_hours / 24.0

                interpretation.append(f"\n  {i}. Período: {period_days:.1f} días ({period_hours:.0f} horas)")
                interpretation.append(f"     Frecuencia: {freq:.4f} Hz")
                interpretation.append(f"     Intensidad: {power:.4f}")

                # Interpretar el período
                period_interpretation = interpret_period(period_days, target_key)
                interpretation.append(f"     📝 {period_interpretation}")
        else:
            interpretation.append("  • No se identificaron patrones dominantes claros")
            interpretation.append("  • La serie puede ser ruidosa o no tener patrones periódicos fuertes")

        # Análisis de estacionalidades esperadas
        interpretation.append(f"\n📅 ANÁLISIS DE ESTACIONALIDADES ESPERADAS:")

        expected_periods = [
            (1, "Variación DIARIA (24h)"),
            (7, "Variación SEMANAL (7d)"),
            (30, "Variación MENSUAL (30d)"),
            (365, "Variación ANUAL (365d)")
        ]

        for period_days, description in expected_periods:
            period_hours = period_days * 24
            expected_freq = 1.0 / period_hours

            # Buscar pico más cercano
            if len(low_freq) > 0:
                idx = np.argmin(np.abs(low_freq - expected_freq))
                closest_freq = low_freq[idx]
                closest_power = low_psd[idx]
                freq_diff = abs(closest_freq - expected_freq)

                # Umbral más flexible para detección
                detection = "✓ DETECTADO" if freq_diff < 0.005 and closest_power > np.percentile(low_psd, 50) else "✗ NO DETECTADO"

                interpretation.append(f"  {description}:")
                interpretation.append(f"    {detection} (diferencia: {freq_diff:.6f} Hz)")

        # Interpretación específica del contaminante
        interpretation.append(f"\n🎯 INTERPRETACIÓN ESPECÍFICA PARA {friendly_name.upper()}:")
        interpretation.extend(interpret_contaminant_patterns(target_key, peaks, low_psd, low_freq if 'low_freq' in locals() else []))

        # Recomendaciones de modelado
        interpretation.append(f"\n💡 RECOMENDACIONES PARA MODELADO:")
        interpretation.extend(generate_modeling_recommendations(peaks, low_psd, low_freq if 'low_freq' in locals() else []))

        # Crear gráfico simplificado
        create_interpretation_plot(friendly_name, series, dates, low_freq, low_psd, peaks, target_key, output_dir)

        return interpretation

    except Exception as e:
        print(f"❌ Error analizando {friendly_name}: {e}")
        import traceback
        traceback.print_exc()
        return [f"❌ Error en análisis de {friendly_name}: {str(e)}"]

def interpret_period(period_days, contaminant_type):
    """Interpreta el significado de un período específico"""

    if period_days < 1:
        return "Patrón intra-diario (variaciones horarias)"
    elif 0.9 <= period_days <= 1.1:
        if contaminant_type == 'no2_median':
            return "Ciclo diario típico de NO2 - relacionado con tráfico vehicular"
        elif contaminant_type == 'co_median':
            return "Ciclo diario de CO - combustión vehicular e industrial"
        else:
            return "Ciclo diario - actividades humanas y condiciones meteorológicas"

    elif 6 <= period_days <= 8:
        if contaminant_type == 'no2_median':
            return "Patrón semanal claro - diferencia días laborables/fin de semana"
        else:
            return "Variación semanal - patrones de actividad humana"

    elif 25 <= period_days <= 35:
        return "Posible patrón mensual - ciclos lunares o nóminas"

    elif 300 <= period_days <= 400:
        return "Estacionalidad ANUAL - condiciones meteorológicas estacionales"

    else:
        return f"Patrón con período de {period_days:.1f} días - verificar causas específicas"

def interpret_contaminant_patterns(contaminant_type, peaks, psd, frequencies):
    """Interpretación específica por tipo de contaminante"""

    interpretation = []

    if contaminant_type == 'pm10_median':
        interpretation.append("  • PM10 - Material Particulado:")
        interpretation.append("    - Fuentes: tráfico, industria, construcción, resuspensión")
        interpretation.append("    - Típicamente muestra patrones diarios y estacionales")
        interpretation.append("    - Sensible a condiciones meteorológicas (viento, lluvia)")
        interpretation.append("    - Puede tener componentes naturales (polvo, sal marina)")

    elif contaminant_type == 'no2_median':
        interpretation.append("  • NO2 - Dióxido de Nitrógeno:")
        interpretation.append("    - Principalmente de fuentes vehiculares")
        interpretation.append("    - Fuertes patrones diarios (horas pico) y semanales")
        interpretation.append("    - Vida atmosférica corta - buen indicador de emisiones recientes")
        interpretation.append("    - Menor estacionalidad anual que PM10")

    elif contaminant_type == 'co_median':
        interpretation.append("  • CO - Monóxido de Carbono:")
        interpretation.append("    - Combustión incompleta de combustibles")
        interpretation.append("    - Patrones similares al NO2 pero con diferente persistencia")
        interpretation.append("    - Acumulación en condiciones de inversión térmica")
        interpretation.append("    - Fuentes: vehículos, industria, quema de biomasa")

    # Análisis de intensidad de patrones
    if len(peaks) > 0 and len(psd) > 0:
        total_power = np.sum(psd[peaks])
        avg_power = total_power / len(peaks)

        if avg_power > np.percentile(psd, 75):
            interpretation.append("  • 📈 PATRONES FUERTES: La serie tiene componentes periódicos bien definidos")
        else:
            interpretation.append("  • 📉 PATRONES DÉBILES: Los componentes periódicos no son dominantes")
    else:
        interpretation.append("  • 🔍 SIN PATRONES CLAROS: La serie puede ser ruidosa o estacionaria")

    return interpretation

def generate_modeling_recommendations(peaks, psd, frequencies):
    """Genera recomendaciones para modelado de series temporales"""

    recommendations = []

    if len(peaks) == 0 or len(frequencies) == 0:
        recommendations.append("  • Considerar modelo ARIMA/SARIMA sin componentes estacionales fuertes")
        recommendations.append("  • Evaluar si la serie es estacionaria antes de modelar")
        recommendations.append("  • Posiblemente la serie es ruidosa o no tiene patrones periódicos claros")
        return recommendations

    # Verificar patrones diarios
    daily_freq = 1.0 / 24.0
    daily_peaks = [i for i in peaks if abs(frequencies[i] - daily_freq) < 0.01]

    if daily_peaks:
        recommendations.append("  • INCLUIR componente DIARIO (24h) en el modelo")

    # Verificar patrones semanales
    weekly_freq = 1.0 / (24.0 * 7)
    weekly_peaks = [i for i in peaks if abs(frequencies[i] - weekly_freq) < 0.001]

    if weekly_peaks:
        recommendations.append("  • INCLUIR componente SEMANAL (7d) en el modelo")

    # Verificar patrones anuales
    yearly_freq = 1.0 / (24.0 * 365)
    yearly_peaks = [i for i in peaks if abs(frequencies[i] - yearly_freq) < 0.0001]

    if yearly_peaks:
        recommendations.append("  • INCLUIR componente ANUAL (365d) en el modelo")

    if not recommendations:
        recommendations.append("  • Considerar modelo ARIMA/SARIMA sin componentes estacionales fuertes")
        recommendations.append("  • Evaluar si la serie es estacionaria antes de modelar")

    recommendations.append("  • Validar componentes estacionales con descomposición STL")
    recommendations.append("  • Considerar efectos externos (meteorología, eventos especiales)")

    return recommendations

def create_interpretation_plot(friendly_name, series, dates, frequencies, psd, peaks, target_key, output_dir):
    """Crea un gráfico de interpretación simplificado"""

    try:
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

        # Panel 1: Serie temporal
        ax1.plot(dates, series, color='blue', alpha=0.7, linewidth=1)
        ax1.set_title(f'{friendly_name} - Serie Temporal', fontsize=14, fontweight='bold')
        ax1.set_ylabel('Concentración')
        ax1.grid(True, alpha=0.3)

        # Panel 2: Espectro de frecuencias
        if len(frequencies) > 0 and len(psd) > 0:
            ax2.plot(frequencies, psd, color='red', linewidth=2)

            # Marcar picos
            if len(peaks) > 0:
                for peak in peaks[:5]:  # Top 5 picos
                    if peak < len(frequencies) and peak < len(psd):
                        freq = frequencies[peak]
                        power = psd[peak]
                        period_days = (1.0 / freq) / 24.0 if freq > 0 else 0

                        ax2.plot(freq, power, 'ro', markersize=8)
                        ax2.annotate(f'{period_days:.1f}d',
                                    xy=(freq, power),
                                    xytext=(10, 10),
                                    textcoords='offset points',
                                    fontsize=9,
                                    bbox=dict(boxstyle="round,pad=0.3", fc="yellow", alpha=0.7))

            # Líneas de referencia para períodos importantes
            important_periods = [1, 7, 30, 365]  # días
            for period in important_periods:
                freq_ref = 1.0 / (period * 24)
                if len(frequencies) > 0 and freq_ref <= frequencies[-1]:
                    ax2.axvline(freq_ref, color='orange', linestyle='--', alpha=0.7, label=f'{period}d')

            ax2.set_title('Espectro de Frecuencias - Patrones Identificados', fontsize=14, fontweight='bold')
            ax2.set_xlabel('Frecuencia (ciclos/hora)')
            ax2.set_ylabel('Densidad Espectral de Potencia')
            ax2.legend()
            ax2.grid(True, alpha=0.3)

        plt.tight_layout()

        # Guardar
        plot_path = os.path.join(output_dir, f"interpretacion_{target_key}.png")
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.close()

        print(f"  📊 Gráfico de interpretación guardado: {plot_path}")

    except Exception as e:
        print(f"  ❌ Error creando gráfico para {friendly_name}: {e}")

# Ejecutar el análisis completo
if __name__ == "__main__":
    report = create_complete_fourier_interpretation()

CREANDO ANÁLISIS FOURIER COMPLETO CON INTERPRETACIÓN
Cargando datos...
Calculando medianas...

Analizando pm10_median...
  📊 Gráfico de interpretación guardado: fourier_analysis_complete/interpretacion_pm10_median.png

Analizando no2_median...
  📊 Gráfico de interpretación guardado: fourier_analysis_complete/interpretacion_no2_median.png

Analizando co_median...
  📊 Gráfico de interpretación guardado: fourier_analysis_complete/interpretacion_co_median.png

✅ REPORTE COMPLETO GUARDADO: fourier_analysis_complete/REPORTE_INTERPRETACION_FOURIER.txt

RESUMEN EJECUTIVO DEL ANÁLISIS FOURIER
REPORTE COMPLETO DE ANÁLISIS FOURIER - INTERPRETACIÓN


🎯 ANÁLISIS DE MATERIAL PARTICULADO PM10

📊 DATOS GENERALES:
  • Muestras analizadas: 125771
  • Frecuencia de muestreo: 1.00 Hz
  • Período total: 5368.0 días
  • Picos espectrales identificados: 2797

🔍 PATRONES TEMPORALES IDENTIFICADOS:

  1. Período: 1746.8 días (41924 horas)
     Frecuencia: 0.0000 Hz
     Intensidad: 296908.4641
     📝 Patrón con